In [1]:
!pip install transformers sentencepiece
!pip install torch

In [2]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")
emotion_model = pipeline("text-classification",
                         model="j-hartmann/emotion-english-distilroberta-base",
                         return_all_scores=False)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [3]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 48.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import json
import spacy
from transformers import pipeline
from collections import Counter

# Load spaCy large English model for NER and POS
nlp = spacy.load("en_core_web_trf")  # better for proper nouns

# Load sentiment classifier
sentiment_classifier = pipeline("sentiment-analysis", device=0)  # use cuda:0

# Function to extract persona from a conversation
def extract_persona(sample):
    content_list = sample.get("content", [])
    full_text = " ".join([c.get("User","") for c in content_list])

    # Run NLP
    doc = nlp(full_text)

    # Extract entities
    entities = {}
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "AGE"]:
            entities.setdefault(ent.label_, []).append(ent.text)

    # Extract main nouns and proper nouns
    nouns = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]]
    noun_counts = Counter(nouns)
    main_nouns = [word for word, count in noun_counts.most_common(30)]

    # Extract personality traits from adjectives
    personality = [token.text for token in doc if token.pos_ == "ADJ"]
    personality_counts = Counter(personality)
    personality_traits = [word for word, count in personality_counts.most_common(20)]

    # Key sentences (top 5 longest or most informative)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    key_sentences = sentences[:8]  # take first 8 for simplicity

    # Sentiment
    sentiment_result = sentiment_classifier(full_text[:512])[0]  # truncate for efficiency
    sentiment = sentiment_result['label']

    # Very basic coping style extraction (can improve later)
    coping_style = []
    if "don't know" in full_text or "not sure" in full_text:
        coping_style.append("uncertain decision-making")
    if "talk" in full_text or "communicate" in full_text:
        coping_style.append("prefers communication when ready")

    # Values (keywords based on context)
    values = []
    for v in ["communication", "relationships", "support", "family", "friendship", "career"]:
        if v in full_text.lower():
            values.append(v)

    # Emotion pattern (basic)
    emotion = "neutral"
    for e in ["fear", "anger", "sad", "happy", "surprise", "disgust", "anxious", "worried"]:
        if e in full_text.lower():
            emotion = e
            break

    persona = {
        "User Name": entities.get("PERSON", ["Unknown"])[0],
        "Age": entities.get("AGE", ["Unknown"])[0],
        "Coping Style": coping_style,
        "Emotion Pattern": emotion,
        "Entities": entities,
        "Key Sentences": key_sentences,
        "Main Nouns": main_nouns,
        "Personality Traits": personality_traits,
        "Sentiment": sentiment,
        "Values": values
    }

    return persona

# Load dataset
dataset_path = "/content/ExTES (1).json"
try:
    with open(dataset_path, "r", encoding="utf-8") as f:
        data = json.load(f)
except json.JSONDecodeError as e:
    print(f"Error loading JSON file: {e}")
    print(f"The file '{dataset_path}' appears to be malformed. Please check line 346588 column 6 for an unquoted property name.")
    data = [] # Assign an empty list to prevent further errors, or handle as appropriate

# Generate persona for first sample
if data:
    first_sample = data[5]
    persona = extract_persona(first_sample)

    import pprint
    pprint.pprint(persona)
else:
    print("Cannot generate persona as the dataset could not be loaded due to JSON errors.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


{'Age': 'Unknown',
 'Coping Style': ['uncertain decision-making',
                  'prefers communication when ready'],
 'Emotion Pattern': 'worried',
 'Entities': {},
 'Key Sentences': ['Hey, are you available?',
                   'I need someone to talk to.',
                   "We've been arguing a lot lately, especially when it comes "
                   'to our future plans.',
                   "It feels like we're on completely different paths and "
                   "can't see eye to eye.",
                   "So, I've been wanting to pursue a career in music, which "
                   'requires me to travel and take risks.',
                   'On the other hand, my friend values stability and wants to '
                   'settle down with a traditional job and a family.',
                   "Honestly, I respect their choices, but it's frustrating "
                   "because they don't seem to understand my passion for "
                   'music.',
                   '

In [5]:
import json
import spacy
from transformers import pipeline
from collections import Counter
from tqdm import tqdm

# Load spaCy large English model for NER and POS
nlp = spacy.load("en_core_web_trf")  # better for proper nouns

# Load sentiment classifier
sentiment_classifier = pipeline("sentiment-analysis", device=0)  # use cuda:0

def extract_persona(sample):
    content_list = sample.get("content", [])
    full_text = " ".join([c.get("User","") for c in content_list])

    # Run NLP
    doc = nlp(full_text)

    # Extract entities
    entities = {}
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "LOC", "DATE", "AGE", "NORP"]:
            entities.setdefault(ent.label_, []).append(ent.text)

    # Extract main nouns and proper nouns
    nouns = [token.text.lower() for token in doc if token.pos_ in ["NOUN", "PROPN"]]
    noun_counts = Counter(nouns)
    main_nouns = [word for word, count in noun_counts.most_common(30)]

    # Extract personality traits from adjectives
    personality = [token.text.lower() for token in doc if token.pos_ == "ADJ"]
    personality_counts = Counter(personality)
    personality_traits = [word for word, count in personality_counts.most_common(20)]

    # Key sentences (top 8 longest by word count)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    sentences_sorted = sorted(sentences, key=lambda x: len(x.split()), reverse=True)
    key_sentences = sentences_sorted[:8]

    # Sentiment
    sentiment_result = sentiment_classifier(full_text[:512])[0]  # truncate for efficiency
    sentiment = sentiment_result['label']

    # Coping style heuristics
    coping_style = []
    text_lower = full_text.lower()
    if any(kw in text_lower for kw in ["don't know", "not sure", "uncertain", "confused"]):
        coping_style.append("uncertain decision-making")
    if any(kw in text_lower for kw in ["talk", "communicate", "share feelings"]):
        coping_style.append("prefers communication when ready")
    if any(kw in text_lower for kw in ["wait", "avoid", "hold back"]):
        coping_style.append("avoidance")
    if any(kw in text_lower for kw in ["think", "reflect", "consider"]):
        coping_style.append("reflective")
    if any(kw in text_lower for kw in ["care", "support", "understand"]):
        coping_style.append("empathetic")

    # Values (keywords based on context)
    values = []
    value_keywords = ["communication", "relationships", "support", "family", "friendship",
                      "career", "trust", "honesty", "wellbeing", "help", "learning"]
    for v in value_keywords:
        if v in text_lower:
            values.append(v)

    # Emotion pattern (basic keyword match)
    emotion = "neutral"
    emotion_keywords = ["fear", "anger", "sad", "happy", "surprise", "disgust", "anxious", "worried"]
    for e in emotion_keywords:
        if e in text_lower:
            emotion = e
            break

    persona = {
        "User Name": entities.get("PERSON", ["Unknown"])[0],
        "Age": entities.get("AGE", ["Unknown"])[0],
        "Coping Style": coping_style,
        "Emotion Pattern": emotion,
        "Entities": entities,
        "Key Sentences": key_sentences,
        "Main Nouns": main_nouns,
        "Personality Traits": personality_traits,
        "Sentiment": sentiment,
        "Values": values
    }

    return persona

# Load dataset
dataset_path = "/content/ExTES (1).json"
with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Process entire dataset
all_personas = []
for sample in tqdm(data, desc="Generating personas"):
    persona = extract_persona(sample)
    all_personas.append(persona)

# Save all personas to JSON
output_path = "/content/ExTES_personas.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_personas, f, ensure_ascii=False, indent=4)

print(f"Saved {len(all_personas)} personas to {output_path}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0
Generating personas: 100%|██████████| 11178/11178 [2:28:38<00:00,  1.25it/s]


Saved 11178 personas to /content/ExTES_personas.json


In [6]:
import json

# Paths
extes_path = "/content/ExTES (1).json"
persona_path = "/content/ExTES_personas.json"
output_path = "/content/ExTES_with_persona.json"

# Load EXTES dataset
with open(extes_path, "r", encoding="utf-8") as f:
    extes_data = json.load(f)

# Load personas dataset
with open(persona_path, "r", encoding="utf-8") as f:
    persona_data = json.load(f)

# Safety check
if len(extes_data) != len(persona_data):
    raise ValueError(f"Length mismatch: EXTES={len(extes_data)}, Personas={len(persona_data)}")

# Combine
combined = []
for sample, persona in zip(extes_data, persona_data):
    new_sample = sample.copy()
    new_sample["persona"] = persona  # attach persona
    combined.append(new_sample)

# Save combined dataset
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=4)

print(f"Saved {len(combined)} samples to {output_path}")

Saved 11178 samples to /content/ExTES_with_persona.json
